# Praktikum 01: Spezifische Wärmekapazität eines Metalls

## Musterlösung (Lehrperson)

Dieses Notebook berechnet aus den Messwerten des Praktikums für **alle vier Metallgruppen** (Aluminium, Nickel, Kupfer, Messing) die spezifische Wärmekapazität $c_M$, führt eine Fehlerrechnung mit den einfachen Regeln für Addition, Subtraktion, Multiplikation und Division durch (keine Ableitungen, angepasst an den Wissensstand der 5. Klasse), vergleicht jedes Resultat mit der Literaturtabelle aus Abschnitt 2.2 der Anleitung und identifiziert das jeweils gemessene Metall über $c_M$ und Dichte. Am Ende exportiert es alle Werte für das automatisch generierte Lösungsblatt (`01BruD_Loesungsblatt.tex`).

Die Variablennamen entsprechen der Schüleranleitung `01BruD_Spezifische_Waermekapazitaet_Metall.tex` (Messprotokoll in Abschnitt 8). Die Metallproben liegen als kleine Pellets vor, nicht als durchgehender Block.

| Grösse | Bedeutung |
|---|---|
| $m_M$ | Masse der Metallprobe |
| $m_K$ | Masse des Kalorimeters (Kupferteil inkl. Rührwerkzeug) |
| $m_W$ | Masse des Wassers im Kalorimeter |
| $T_{M0}$ | Anfangstemperatur der Metallprobe (Mittel aus $T_{WB}$ und $T_{KT}$) |
| $T_0 = T_{W0} = T_{K0}$ | Anfangstemperatur von Wasser und Kalorimeter |
| $T_1 = T_{W1} = T_{K1} = T_{M1}$ | Mischtemperatur |
| $c_W$, $c_K$ | spezifische Wärmekapazität von Wasser bzw. Kupfer (Kalorimetergefäss) |

In der Zelle unter "1. Messwerte eingeben" für jede der vier Gruppen (Aluminium/Nickel/Kupfer/Messing) die tatsächlich gemessenen Werte eintragen. Die vorhandenen Zahlen sind Platzhalter, mit denen sich das Notebook durchrechnen lässt, um den Ablauf zu prüfen; sie sind so gewählt, dass jede Gruppe rechnerisch ihr eigenes Metall korrekt identifiziert. Hat eine Gruppe kein Volumen gemessen, `"V_M": None` eintragen, die Dichtebestimmung wird dann für diese Gruppe übersprungen.

**Wichtig für den PDF-Export:** Damit die letzte Zelle (Abschnitt 7) die Datei `loesung_werte.tex` am richtigen Ort ablegt, muss der Kernel im selben Ordner wie dieses Notebook laufen (in VS Code ist das der Standard). Nach jeder Änderung der Messwerte in Abschnitt 1: Notebook komplett neu laufen lassen ("Run All"), danach `01BruD_Loesungsblatt.tex` einmal mit pdflatex (zweimal) neu kompilieren, siehe Hinweis in Abschnitt 7.

In [ ]:
import sympy as sp
import matplotlib.pyplot as plt

sp.init_printing()

## 1. Messwerte eingeben

In [ ]:
# spezifische Waermekapazitaeten (Literaturwerte, als exakt angenommen) [J/(kg*K)], gelten fuer alle Gruppen
c_W_val = 4190.0   # Wasser
c_K_val = 385.0    # Kupfer (Kalorimetergefaess)

# Messwerte pro Gruppe: (Wert, Messgenauigkeit). Massen in kg, Temperaturen in Grad C, Volumen in m^3.
# V_M auf None setzen, falls diese Gruppe keine Dichtebestimmung gemacht hat.
messungen = {
    "Aluminium": {
        "m_M": (0.100, 0.001), "m_K": (0.200, 0.001), "m_W": (0.100, 0.001),
        "theta_M0": (95.0, 1.0), "theta_0": (20.0, 0.5), "theta_1": (31.5, 0.5),
        "V_M": (3.70e-5, 0.20e-5),
    },
    "Nickel": {
        "m_M": (0.100, 0.001), "m_K": (0.200, 0.001), "m_W": (0.100, 0.001),
        "theta_M0": (95.0, 1.0), "theta_0": (20.0, 0.5), "theta_1": (26.2, 0.5),
        "V_M": (1.123e-5, 0.02e-5),
    },
    "Kupfer": {
        "m_M": (0.100, 0.001), "m_K": (0.200, 0.001), "m_W": (0.100, 0.001),
        "theta_M0": (95.0, 1.0), "theta_0": (20.0, 0.5), "theta_1": (25.4, 0.5),
        "V_M": (1.116e-5, 0.02e-5),
    },
    "Messing": {
        "m_M": (0.100, 0.001), "m_K": (0.200, 0.001), "m_W": (0.100, 0.001),
        "theta_M0": (95.0, 1.0), "theta_0": (20.0, 0.5), "theta_1": (25.3, 0.5),
        "V_M": (1.176e-5, 0.02e-5),
    },
}

## 2. Herleitung der Gleichung für $c_M$ (Energieerhaltung)

Beim Vermischen im Kalorimeter gilt Energieerhaltung: Die von der Metallprobe abgegebene Wärme entspricht der vom Wasser und vom Kalorimetergefäss gemeinsam aufgenommenen Wärme.

$$Q_{abgegeben} = Q_{aufgenommen}$$
$$m_M \, c_M \, (T_{M0}-T_1) = m_W \, c_W \, (T_1-T_0) + m_K \, c_K \, (T_1-T_0)$$

Aufgelöst nach $c_M$ ergibt sich die Gleichung, die die Schülerinnen und Schüler in Abschnitt 9 der Anleitung selbst herleiten sollen. sympy übernimmt diesen Schritt zur Kontrolle. Diese eine Gleichung gilt für alle vier Gruppen gleichermassen, nur die eingesetzten Messwerte unterscheiden sich:

In [ ]:
m_M, m_K, m_W = sp.symbols('m_M m_K m_W', positive=True)
theta_M0, theta_0, theta_1 = sp.symbols('theta_M0 theta_0 theta_1', real=True)
c_W, c_K, c_M = sp.symbols('c_W c_K c_M', positive=True)

energiebilanz = sp.Eq(m_M * c_M * (theta_M0 - theta_1),
                      (m_W * c_W + m_K * c_K) * (theta_1 - theta_0))

cM_formel = sp.solve(energiebilanz, c_M)[0]

display(energiebilanz)
display(sp.Eq(c_M, cM_formel))

## 3. Berechnung für jede Gruppe

Eine Funktion `berechne_alles` wendet die Gleichung aus Abschnitt 2 sowie die Fehlerrechnung aus Abschnitt 4 auf die Messwerte einer einzelnen Gruppe an. Damit lassen sich alle vier Gruppen mit demselben Code auswerten, statt denselben Ablauf viermal zu wiederholen.

In [ ]:
def berechne_alles(d):
    m_M_val, dm_M = d["m_M"]
    m_K_val, dm_K = d["m_K"]
    m_W_val, dm_W = d["m_W"]
    theta_M0_val, dtheta_M0 = d["theta_M0"]
    theta_0_val, dtheta_0 = d["theta_0"]
    theta_1_val, dtheta_1 = d["theta_1"]

    # c_M ueber die hergeleitete Formel (Abschnitt 2)
    werte = {m_M: m_M_val, m_K: m_K_val, m_W: m_W_val,
             theta_M0: theta_M0_val, theta_0: theta_0_val, theta_1: theta_1_val,
             c_W: c_W_val, c_K: c_K_val}
    cM_wert = float(cM_formel.subs(werte))

    # Fehlerrechnung ueber die Zwischengroessen N, Tauf, Tab (Abschnitt 4)
    N_val = m_W_val * c_W_val + m_K_val * c_K_val
    dN = c_W_val * dm_W + c_K_val * dm_K
    T1_val = theta_1_val - theta_0_val
    dT1 = dtheta_1 + dtheta_0
    T2_val = theta_M0_val - theta_1_val
    dT2 = dtheta_M0 + dtheta_1

    rel_N = dN / N_val
    rel_T1 = dT1 / T1_val
    rel_mM = dm_M / m_M_val
    rel_T2 = dT2 / T2_val
    rel_cM = rel_N + rel_T1 + rel_mM + rel_T2
    dcM_wert = cM_wert * rel_cM

    ergebnis = dict(
        m_M_val=m_M_val, dm_M=dm_M, m_K_val=m_K_val, dm_K=dm_K, m_W_val=m_W_val, dm_W=dm_W,
        theta_M0_val=theta_M0_val, dtheta_M0=dtheta_M0, theta_0_val=theta_0_val, dtheta_0=dtheta_0,
        theta_1_val=theta_1_val, dtheta_1=dtheta_1,
        N_val=N_val, dN=dN, T1_val=T1_val, dT1=dT1, T2_val=T2_val, dT2=dT2,
        rel_N=rel_N, rel_T1=rel_T1, rel_mM=rel_mM, rel_T2=rel_T2, rel_cM=rel_cM,
        cM_wert=cM_wert, dcM_wert=dcM_wert,
    )

    # Dichte, falls diese Gruppe ein Volumen gemessen hat (Bonus, Abschnitt 6)
    if d.get("V_M") is not None:
        V_M_val, dV_M = d["V_M"]
        rho_M = m_M_val / V_M_val
        rel_mM_dichte = dm_M / m_M_val
        rel_VM = dV_M / V_M_val
        rel_rho = rel_mM_dichte + rel_VM
        drho_M = rho_M * rel_rho
        ergebnis.update(V_M_val=V_M_val, dV_M=dV_M, rho_M=rho_M, drho_M=drho_M,
                         rel_mM_dichte=rel_mM_dichte, rel_VM=rel_VM, rel_rho=rel_rho)
    return ergebnis


ergebnisse = {name: berechne_alles(daten) for name, daten in messungen.items()}

for name, e in ergebnisse.items():
    print(f"{name:10s}: c_M = ({e['cM_wert']:.1f} +/- {e['dcM_wert']:.1f}) J/(kg*K)")

## 4. Fehlerrechnung (einfache Fehlerfortpflanzung)

Ableitungen sind an dieser Stelle im Unterricht noch nicht behandelt worden. Wir verwenden deshalb die einfachen Regeln für die Fehlerfortpflanzung bei den Grundrechenarten. Für zwei Messgrössen $a$, $b$ mit Messungenauigkeiten $\Delta a$, $\Delta b$:

**Addition/Subtraktion** ($z=a+b$ oder $z=a-b$): die absoluten Fehler addieren sich,
$$\Delta z = \Delta a + \Delta b.$$

**Multiplikation/Division** ($z=a\cdot b$ oder $z=a/b$): die relativen Fehler addieren sich,
$$\frac{\Delta z}{z} = \frac{\Delta a}{a} + \frac{\Delta b}{b}.$$

**Spezialfall Konstante** ($k$ exakt bekannt, z. B. $c_W$, $c_K$): $\Delta(k\cdot a) = k\cdot \Delta a$.

Um diese Regeln auf $c_M = \dfrac{(m_Wc_W+m_Kc_K)(T_1-T_0)}{m_M(T_{M0}-T_1)}$ anzuwenden, zerlegt `berechne_alles` (Abschnitt 3) die Gleichung in Zwischengrössen. Die beiden Temperaturdifferenzen heissen $T_{auf}$ (Erwärmung von Wasser/Kalorimeter) und $T_{ab}$ (Abkühlung der Probe), um sie von den eigentlichen Temperaturen $T_0$, $T_1$, $T_{M0}$ zu unterscheiden:

$$N := m_W c_W + m_K c_K, \qquad T_{auf} := T_1-T_0, \qquad T_{ab} := T_{M0}-T_1, \qquad c_M = \frac{N\cdot T_{auf}}{m_M\cdot T_{ab}}.$$

$N$ wird über die Additionsregel berechnet (jeder Summand zuerst über den Konstanten-Spezialfall), $T_{auf}$ und $T_{ab}$ über die Subtraktionsregel, und $c_M$ selbst ist ein Produkt/Quotient aus vier Faktoren ($N$, $T_{auf}$, $m_M$, $T_{ab}$), also addieren sich deren vier relative Fehler. Im Detail, für jede Gruppe:

In [ ]:
for name, e in ergebnisse.items():
    print(f"=== {name} ===")
    print(f"  N     = {e['N_val']:.2f} J/K,  Delta N     = {e['dN']:.3f} J/K")
    print(f"  Tauf  = {e['T1_val']:.2f} K,   Delta Tauf  = {e['dT1']:.3f} K")
    print(f"  Tab   = {e['T2_val']:.2f} K,   Delta Tab   = {e['dT2']:.3f} K")
    print(f"  relativer Fehler: {e['rel_N']:.4f} (N) + {e['rel_T1']:.4f} (Tauf) + "
          f"{e['rel_mM']:.4f} (m_M) + {e['rel_T2']:.4f} (Tab) = {e['rel_cM']:.4f} ({100*e['rel_cM']:.1f} %)")
    print(f"  c_M = ({e['cM_wert']:.1f} +/- {e['dcM_wert']:.1f}) J/(kg*K)\n")

## 5. Vergleich mit Literaturwerten und Identifikation des Metalls

Literaturwerte für die spezifische Wärmekapazität aus Abschnitt 2.2 der Anleitung:

In [ ]:
literatur_c = {
    "Aluminium": 897,
    "Nickel": 444,
    "Kupfer": 385,
    "Messing": 380,
}

for name, e in ergebnisse.items():
    cM_wert, dcM_wert = e["cM_wert"], e["dcM_wert"]
    print(f"=== Gruppe {name} (gemessen: c_M = {cM_wert:.1f} J/(kg*K)) ===")
    print(f"{'Metall':12s}{'c (Lit.)':>10s}{'Abweichung':>14s}{'rel. Abw.':>12s}{'im Fehlerbereich?':>20s}")
    for metall, c_lit in sorted(literatur_c.items(), key=lambda kv: abs(kv[1]-cM_wert)):
        abweichung = cM_wert - c_lit
        rel_abweichung = 100 * abweichung / c_lit
        im_bereich = abs(abweichung) <= dcM_wert
        print(f"{metall:12s}{c_lit:10.0f}{abweichung:+14.1f}{rel_abweichung:+11.1f} %{str(im_bereich):>20s}")

    vermutet = min(literatur_c, key=lambda m: abs(literatur_c[m]-cM_wert))
    e["vermutetes_metall"] = vermutet
    print(f"--> Beste Uebereinstimmung: {vermutet} (c_Lit = {literatur_c[vermutet]} J/(kg*K))\n")

## 6. Diagramm: alle vier Gruppen mit Fehlerbalken vs. Literaturwerte

In [ ]:
metalle = list(literatur_c.keys())
c_werte_lit = [literatur_c[m] for m in metalle]
farben = {"Aluminium": "crimson", "Nickel": "steelblue", "Kupfer": "darkorange", "Messing": "seagreen"}

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(metalle, c_werte_lit, color="lightgray", edgecolor="black", label="Literaturwerte")

for name, e in ergebnisse.items():
    ax.errorbar(e["vermutetes_metall"], e["cM_wert"], yerr=e["dcM_wert"], fmt="o",
                color=farben.get(name, "black"), capsize=6, markersize=8,
                label=f"Gruppe {name}: cM=({e['cM_wert']:.0f}+/-{e['dcM_wert']:.0f})")

ax.set_ylabel("spez. Waermekapazitaet c in J/(kg*K)")
ax.set_title("Vergleich aller vier Gruppen mit den Literaturwerten")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Bonus: Dichtebestimmung als zweite Bestätigung

Die Anleitung sieht in der Auswertung zusätzlich eine Dichtebestimmung vor, um die Vermutung aus der Wärmekapazität zu bestätigen (Fehlerrechnung dafür ist eine eigene Auswertungsfrage). Für jede Gruppe mit eingetragenem Volumen wird hier die Dichte berechnet und mit der Literatur verglichen, über dieselbe Multiplikations-/Divisionsregel wie in Abschnitt 4.

In [ ]:
literatur_rho = {   # typische Literaturwerte, in kg/m^3
    "Aluminium": 2700,
    "Nickel": 8908,
    "Kupfer": 8960,
    "Messing": 8500,
}

for name, e in ergebnisse.items():
    if "rho_M" not in e:
        print(f"{name}: kein Volumen eingetragen, Dichtebestimmung wird uebersprungen.\n")
        continue

    rho_M, drho_M = e["rho_M"], e["drho_M"]
    print(f"=== Gruppe {name} (gemessen: rho_M = {rho_M:.0f} +/- {drho_M:.0f} kg/m^3) ===")
    print(f"relativer Fehler: {e['rel_mM_dichte']:.4f} (m_M) + {e['rel_VM']:.4f} (V_M) "
          f"= {e['rel_rho']:.4f} ({100*e['rel_rho']:.1f} %)")
    print(f"{'Metall':12s}{'rho (Lit.)':>12s}{'rel. Abw.':>12s}")
    for metall, rho_lit in sorted(literatur_rho.items(), key=lambda kv: abs(kv[1]-rho_M)):
        rel_abw = 100 * (rho_M - rho_lit) / rho_lit
        print(f"{metall:12s}{rho_lit:12.0f}{rel_abw:+11.1f} %")

    vermutet_dichte = min(literatur_rho, key=lambda m: abs(literatur_rho[m]-rho_M))
    e["vermutetes_metall_dichte"] = vermutet_dichte
    print(f"--> Beste Uebereinstimmung (Dichte): {vermutet_dichte}")
    if vermutet_dichte == e["vermutetes_metall"]:
        print("Waermekapazitaet und Dichte stimmen im selben Metall ueberein.\n")
    else:
        print("Waermekapazitaet und Dichte deuten auf unterschiedliche Metalle hin, Messung pruefen.\n")

## 7. Export für das Lösungsblatt (PDF)

Diese Zelle schreibt alle Werte (Messwerte, Zwischengrössen, Fehler, Ergebnis) für **jede der vier Gruppen** als `\newcommand`-Definitionen in die Datei `loesung_werte.tex`. Damit die Makronamen eindeutig bleiben, wird der Metallname an jeden Namen angehängt (z. B. `\cMwertNickel` statt `\cMwert`). Diese Datei wird von `01BruD_Loesungsblatt.tex` per `\input` eingebunden, sodass das Lösungsblatt beim nächsten Kompilieren automatisch die aktuellen Werte zeigt. Ablauf bei geänderten Messwerten: Notebook neu laufen lassen ("Run All") → `01BruD_Loesungsblatt.tex` mit pdflatex (zweimal) neu kompilieren.

`\newcommand`-Namen dürfen in LaTeX nur aus Buchstaben bestehen, deshalb heissen die internen Makro-Schlüssel weiterhin `theta...`/`T...` (z. B. `\thetaEinsNickel` für die Mischtemperatur $T_1$ von Nickel), auch wenn im Lösungsblatt selbst überall $T$ statt $\vartheta$ angezeigt wird. Massen werden in Gramm, Volumen in cm³ (ml) und Dichte in g/cm³ exportiert, das ist für ein Praktikumsprotokoll die gewohntere Grössenordnung als kg, m³ und kg/m³.

In [ ]:
import pathlib
import math

def fmt(x, nachkommastellen=1):
    return f"{x:.{nachkommastellen}f}"

def sci_tex(x, sig=1):
    """Formatiert x als LaTeX-Wissenschaftsnotation, z.B. 3.7e-05 -> '3.7\\times10^{-5}'."""
    if x == 0:
        return "0"
    exponent = math.floor(math.log10(abs(x)))
    mantisse_str = f"{x / (10 ** exponent):.{sig}f}"
    if abs(float(mantisse_str)) >= 10:   # z.B. 9.96 wird zu "10.0" gerundet -> Exponent nachziehen
        exponent += 1
        mantisse_str = f"{x / (10 ** exponent):.{sig}f}"
    return f"{mantisse_str}\\times10^{{{exponent}}}"

alle_export = {}
for name, e in ergebnisse.items():
    px = {
        f"mM{name}": fmt(e["m_M_val"] * 1000, 0), f"dmM{name}": fmt(e["dm_M"] * 1000, 0),
        f"mK{name}": fmt(e["m_K_val"] * 1000, 0), f"dmK{name}": fmt(e["dm_K"] * 1000, 0),
        f"mW{name}": fmt(e["m_W_val"] * 1000, 0), f"dmW{name}": fmt(e["dm_W"] * 1000, 0),
        f"mMkg{name}": fmt(e["m_M_val"], 3), f"mKkg{name}": fmt(e["m_K_val"], 3), f"mWkg{name}": fmt(e["m_W_val"], 3),
        f"thetaMNull{name}": fmt(e["theta_M0_val"], 1), f"dthetaMNull{name}": fmt(e["dtheta_M0"], 1),
        f"thetaNull{name}": fmt(e["theta_0_val"], 1), f"dthetaNull{name}": fmt(e["dtheta_0"], 1),
        f"thetaEins{name}": fmt(e["theta_1_val"], 1), f"dthetaEins{name}": fmt(e["dtheta_1"], 1),
        f"cW{name}": fmt(c_W_val, 0), f"cK{name}": fmt(c_K_val, 0),
        f"mWcW{name}": fmt(e["m_W_val"] * c_W_val, 1), f"mKcK{name}": fmt(e["m_K_val"] * c_K_val, 1),
        f"Nwert{name}": fmt(e["N_val"], 1), f"dNwert{name}": fmt(e["dN"], 1),
        f"Teins{name}": fmt(e["T1_val"], 1), f"dTeins{name}": fmt(e["dT1"], 1),
        f"Tzwei{name}": fmt(e["T2_val"], 1), f"dTzwei{name}": fmt(e["dT2"], 1),
        f"mMTzwei{name}": fmt(e["m_M_val"] * e["T2_val"], 2), f"NTeins{name}": fmt(e["N_val"] * e["T1_val"], 1),
        f"relN{name}": fmt(100 * e["rel_N"], 1), f"relTeins{name}": fmt(100 * e["rel_T1"], 1),
        f"relmM{name}": fmt(100 * e["rel_mM"], 1), f"relTzwei{name}": fmt(100 * e["rel_T2"], 1),
        f"relcM{name}": fmt(100 * e["rel_cM"], 1),
        f"cMwert{name}": fmt(e["cM_wert"], 1), f"dcMwert{name}": fmt(e["dcM_wert"], 1),
        f"vermutetesMetall{name}": e["vermutetes_metall"],
        f"cLit{name}": fmt(literatur_c[e["vermutetes_metall"]], 0),
    }
    if "rho_M" in e:
        px.update({
            f"VM{name}": fmt(e["V_M_val"] * 1e6, 1), f"dVM{name}": fmt(e["dV_M"] * 1e6, 1),
            f"VMSI{name}": sci_tex(e["V_M_val"]), f"dVMSI{name}": sci_tex(e["dV_M"]),
            f"relmMDichte{name}": fmt(100 * e["rel_mM_dichte"], 1), f"relVM{name}": fmt(100 * e["rel_VM"], 1),
            f"relrho{name}": fmt(100 * e["rel_rho"], 1),
            f"rhoM{name}": fmt(e["rho_M"] / 1000, 2), f"drhoM{name}": fmt(e["drho_M"] / 1000, 2),
            f"rhoMSI{name}": fmt(e["rho_M"], 0),
            f"rhoLit{name}": fmt(literatur_rho[e["vermutetes_metall_dichte"]] / 1000, 2),
            f"vermutetesMetallDichte{name}": e["vermutetes_metall_dichte"],
        })
    else:
        for key in ["VM", "dVM", "VMSI", "dVMSI", "relmMDichte", "relVM", "relrho",
                    "rhoM", "drhoM", "rhoMSI", "rhoLit", "vermutetesMetallDichte"]:
            px[f"{key}{name}"] = "---"
    alle_export.update(px)

pfad = pathlib.Path("loesung_werte.tex")
with pfad.open("w", encoding="utf-8") as f:
    for key, wert in alle_export.items():
        f.write(f"\\newcommand{{\\{key}}}{{{wert}}}\n")

print(f"{len(alle_export)} Werte fuer {len(ergebnisse)} Gruppen nach {pfad.resolve()} exportiert.")

## Zusammenfassung

Resultat je Gruppe: $c_M$ mit Fehlerbereich (Abschnitt 4), vermutetes Metall (Abschnitt 5), Dichte mit Fehlerbereich, falls vorhanden (Bonus). Diese Werte lassen sich direkt in Abschnitt "Mein Ergebnis" der jeweiligen Schüleranleitung als Musterlösung übertragen.